# Topic05 — Which request gets the next token?

## Your investigation

You route support tickets with Qwen2.5-0.5B-Instruct. `I forgot my password and cannot sign in to my account.` has gold label `access`; `I can log in, but clicking Export crashes the application.` has gold label `bug`. Gold is the required class, not a promised model answer.

Implement two small CPU functions, predict scheduling behavior, then compare submitted bursts 1 and 4 in two inference stacks. Inspect literal answers and class errors before choosing a faster route. Timing statistics and runtime infrastructure are provided.

## Before starting

Select T4, High RAM OFF before setup. CPU work comes first. GPU packages/model loading wait for your completed work, saved prediction and explicit `RUN_GPU=True` in Task 3. Free GPU quota is not guaranteed; a CPU-only route is not full lab completion.

Everything needed is prepared by this notebook: public inputs are downloaded automatically at a fixed commit and SHA-256 checked. No local Python file, manual file upload, Drive mount or private GitHub login is needed. Internet is required for setup/model inputs. Keep `%%lsm` on scientific cells: managed Python 3.11.17 uses frozen dependencies rather than Colab's inherited scientific libraries.

## Completion

Pass the two CPU test cells, retain both traces, complete your explanation/prediction, obtain 144 predictions and 90 cohort intervals, read the graphs/errors and save the four-field decision plus actual attempt ZIP and modified notebook. Viewers never rerun inference.

**Review draft:** the same GPU core has an inspected owner-run on 14 October 2026. This changed student template/new viewers do not yet have their own exact browser acceptance or learner pilot.


## Reproducible execution environment

The Colab kernel forwards each visible `%%lsm` cell to one managed Python
3.11.17 worker. Scientific imports and your functions live in that worker,
not in Colab's provider namespace. Keep `%%lsm` as the first line when adding
a scientific cell. Errors remain visible; there is no automatic retry.
This isolates dependencies, not the OS, driver or untrusted code.

First verify the supplied inputs, then install the frozen CPU profile, then
start the worker. CPU setup installs no Torch, vLLM or bitsandbytes. GPU
installation remains at the topic's explicit GPU stage. Free GPU quota is
not guaranteed.


In [ ]:
def download_verified(url: str, sha256: str, max_bytes: int) -> bytes:
    """Download one frozen public input without executing it.

    Parameters
    ----------
    url : str
        Public HTTPS file at an immutable course commit; no credentials.
    sha256 : str
        Exact expected SHA-256 of the whole file.
    max_bytes : int
        Positive byte limit; a larger response is rejected.

    Returns
    -------
    bytes
        Verified bytes, not an imported module. Nothing is uploaded.

    Raises
    ------
    ValueError
        Wrong protocol, byte limit, size or digest. Network errors propagate.
    """
    import hashlib
    from urllib.request import urlopen

    if not url.startswith("https://") or max_bytes <= 0:
        raise ValueError("An HTTPS input and positive byte limit are required")
    with urlopen(url, timeout=60) as response:
        data = response.read(max_bytes + 1)
    if len(data) > max_bytes or hashlib.sha256(data).hexdigest() != sha256:
        raise ValueError("Frozen input size/digest mismatch: " + url)
    return data


In [ ]:
import hashlib, io, zipfile, uuid
from pathlib import Path

if globals().get("bridge") is not None and bridge.process.poll() is None:
    raise RuntimeError("Preserve results and close this owned worker before new setup")
RUNTIME_URL = "https://raw.githubusercontent.com/levshaazz/llm-serving-mastery-submission-example/d81c299ac1c8fe6249f8b33fa49cd7d42add42ea/public/downloads/notebook-runtime-v1.zip"
RUNTIME_SHA = "64a304935e0a20428d8d6a717d49f021bb64e1a91a4856d4e58b0eb0bcc76e36"
payload = download_verified(RUNTIME_URL, RUNTIME_SHA, 1024 * 1024)
RUNTIME_ROOT = Path("/content/lsm-notebook-runtime-" + uuid.uuid4().hex)
RUNTIME_ROOT.mkdir(exist_ok=False)
expected = {
    "bridge.py": "43895d5a988621f46f124d256a548d46ac07b134858a90acd9b192cb0b615826",
    "pyproject.toml": "189a229da208f06e0b2675db84376bc50710fe066602a017d8dedde4bc3559bc",
    "runtime.py": "097b5f36ccaa602bf0263cd0a2e875aa9a7081c40e6657607149961b1f68f8fa",
    "uv.lock": "b7afd87078d41bff9753d5c54e7e7d58326baeafb93641ee3949e0ee799c3885",
    "worker.py": "6cb5061f1275cf672c58ebf786ddac77bb22b29fed4f8b705e2cae7e073accfe",
}
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == set(expected) | {"manifest.json"}
    for name in archive.namelist():
        assert "/" not in name and "\\" not in name and name not in (".", "..")
        data = archive.read(name)
        if name in expected:
            assert hashlib.sha256(data).hexdigest() == expected[name], name
        with (RUNTIME_ROOT / name).open("xb") as destination:
            destination.write(data)
print("Runtime inputs verified; no packages or models installed")


### Install the frozen CPU profile

The report names the actual managed executable/version/hash and import paths. Installer logs remain in the fresh attempt directory. No base scientific packages are inherited.


In [ ]:
import json
import importlib.util

runtime_spec = importlib.util.spec_from_file_location(
    "lsm_runtime", RUNTIME_ROOT / "runtime.py"
)
runtime = importlib.util.module_from_spec(runtime_spec)
runtime_spec.loader.exec_module(runtime)
RUNTIME_PYTHON = runtime.install_runtime(RUNTIME_ROOT, "cpu")
print(
    json.dumps(json.loads((RUNTIME_ROOT / "identity-cpu.json").read_text()), indent=2)
)


### Start one persistent scientific namespace

The following provided cell registers the literal-code transport. Existing variables remain available across `%%lsm` cells; the provider and worker namespaces are separate.


In [ ]:
bridge_spec = importlib.util.spec_from_file_location(
    "lsm_bridge", RUNTIME_ROOT / "bridge.py"
)
bridge_module = importlib.util.module_from_spec(bridge_spec)
bridge_spec.loader.exec_module(bridge_module)
bridge = bridge_module.IsolatedCellBridge(RUNTIME_PYTHON, RUNTIME_ROOT / "worker.py")


In [ ]:
def run_isolated(line: str, cell: str) -> None:
    """Execute literal cell code in the owned managed worker.

    Parameters
    ----------
    line : str
        Magic arguments; empty for this transport.
    cell : str
        Actual visible Python source, not a precomputed output.

    Returns
    -------
    None
        Renders actual output and propagates failures.
    """
    if line.strip():
        raise ValueError("This transport accepts no magic arguments")
    bridge.render(cell)


In [ ]:
get_ipython().register_magic_function(run_isolated, "cell", "lsm")
print("One managed worker registered; no GPU package imported")


In [ ]:
HELPER_URL = "https://raw.githubusercontent.com/levshaazz/llm-serving-mastery-submission-example/d81c299ac1c8fe6249f8b33fa49cd7d42add42ea/teaching/topic05_scheduling.py"
HELPER_SHA = "8508828a09f3068d548db2e749f473350dd0da3826fdc15959b1efc16b45a1bb"
helper_bytes = download_verified(HELPER_URL, HELPER_SHA, 128 * 1024)
helper_path = RUNTIME_ROOT / "workspace" / "topic05_scheduling.py"
if helper_path.exists():
    assert hashlib.sha256(helper_path.read_bytes()).hexdigest() == HELPER_SHA
else:
    with helper_path.open("xb") as destination:
        destination.write(helper_bytes)
print("Verified public helper:", helper_path.name, "; no inference")


In [ ]:
%%lsm
from pathlib import Path
import hashlib, importlib.util
from typing import Sequence

HELPER = Path("topic05_scheduling.py")
spec = importlib.util.spec_from_file_location("topic05_scheduling", HELPER)
infra = importlib.util.module_from_spec(spec)
spec.loader.exec_module(infra)
KVAccount = infra.KVAccount
Request = infra.Request


In [ ]:
%%lsm
from html import escape
from typing import Mapping, Sequence


def format_cell_value(value: object) -> str:
    """Format a scalar or short vector for display without changing stored data.

    Parameters
    ----------
    value : object
        Scalar, list or tuple; floats are shown with six significant digits.
        Nested lists preserve their row boundaries. None denotes a missing value.

    Returns
    -------
    str
        Display text only; original numerical values remain unchanged.

    Raises
    ------
    ValueError
        If a mapping is supplied as one cell. Flatten named metrics into columns
        or separate tables instead of displaying a dense nested result object.
    """
    if value is None:
        return "—"
    if isinstance(value, Mapping):
        raise ValueError(
            "A table cell cannot contain a result dictionary: use named scalar columns"
        )
    if isinstance(value, float):
        return format(value, ".6g")
    if isinstance(value, (list, tuple)):
        return "[" + ", ".join(format_cell_value(item) for item in value) + "]"
    return str(value)


In [ ]:
%%lsm
from html import escape
from typing import Mapping, Sequence


def show_table(
    rows: Sequence[Mapping[str, object]], columns: Sequence[str], caption: str
) -> None:
    """Display one escaped, labelled table; never rerun a measurement.

    Parameters
    ----------
    rows : Sequence[Mapping[str, object]]
        Records, including an empty population. Missing/None displays a dash.
        Nested dictionaries are rejected. Stored values are never changed.
    columns : Sequence[str]
        Scalar field order; put units in headers when relevant.
    caption : str
        Question, population and actual/toy evidence scope.

    Returns
    -------
    None
        One horizontally bounded HTML table. Floats display six significant
        digits; literal ticket/output text wraps without changing its content.
    """
    import pandas as pd
    from IPython.display import HTML, display

    values = [
        [format_cell_value(row.get(key)) + "\u00a0\u00a0" for key in columns]
        for row in rows
    ]
    headers = [name.replace("_", " ") + "\u00a0\u00a0" for name in columns]
    table = pd.DataFrame(values, columns=headers).to_html(
        index=False, escape=True, border=0
    )
    label = (
        "<caption>"
        + escape(caption)
        + "<br><small>Display: six significant digits; stored values unchanged.</small></caption>"
    )
    table = table.replace("<thead>", label + "<thead>", 1)
    style = "<style>.lsm-table-region td,.lsm-table-region th{padding:.4em .6em;text-align:left;vertical-align:top;max-width:32em;overflow-wrap:anywhere;border-bottom:1px solid #ccc}</style>"
    display(
        HTML(
            style
            + '<div class="lsm-table-region" style="max-width:100%;overflow-x:auto">'
            + table
            + "</div>"
        )
    )


In [ ]:
%%lsm
from html import escape
from typing import Mapping, Sequence


def show_text(title: str, text: str) -> None:
    """Display one literal input/output or diagnostic text artifact.

    Parameters
    ----------
    title : str
        Case identity and evidence scope.
    text : str
        Literal content; HTML is escaped, not interpreted.

    Returns
    -------
    None
        One readable text panel; no GPU call, write or state change.
    """
    from IPython.display import HTML, display

    display(
        HTML(
            "<h4>"
            + escape(title)
            + '</h4><pre style="white-space:pre-wrap">'
            + escape(text)
            + "</pre>"
        )
    )


## A1. KV payload is not model-weight memory

For Qwen2.5-0.5B-Instruct: 24 full-attention layers, 14 query heads, 2 KV heads, head width 64, FP16 elements 2 bytes. One retained position contains K AND V in every layer. Logical per-layer K and V each have axes [KV head,position,feature]. Explain why using 14 query heads would overcount KV by 7. Predict bytes per position before running. This calculation excludes block tables, alignment and all weights.


In [ ]:
%%lsm
BYTES_PER_POSITION = infra.kv_bytes_per_position(**infra.QWEN)
print(
    "Logical KV payload:",
    BYTES_PER_POSITION,
    "bytes/position =",
    BYTES_PER_POSITION / 1024,
    "KiB/position",
)


## A2. Round each independent request

Lengths [17,31,33] mean retained positions, not words or future output caps. Block size 16 means 16 positions per block. Requests do not share blocks in this exercise. Predict blocks and unused positions. Why can ceil(sum(lengths)/16) not represent separate block tables? Implement the contract below; no GPU needed.


### Task 1 — implement request-local KV accounting

Change only the TODO body of `account_kv`, preserving its types and NumPy docstring. Return exactly the six named fields in its contract. Counts refer to retained token positions; byte fields refer to K AND V across all model layers. Do not count query heads as KV heads.

Before running, write your predicted block count for `[17,31,33]` with block size 16. Explain the request boundary. Run the next fixture cell, then inspect the table and block-size plot. Check empty input and zero-length requests; neither creates a block.


In [ ]:
%%lsm
def account_kv(
    lengths: Sequence[int], block_size: int, bytes_per_position: int
) -> KVAccount:
    """Account for separately allocated request KV blocks.

    Parameters
    ----------
    lengths : Sequence[int]
        Nonnegative retained-position counts; zero uses no blocks. No sharing.
    block_size : int
        Positive position slots per physical block.
    bytes_per_position : int
        Positive KV payload bytes across all relevant layers.

    Returns
    -------
    KVAccount
        blocks; live_positions; capacity_positions; unused_positions;
        payload_bytes; capacity_bytes. Empty lengths produce all zeros.
        Excludes metadata, allocator reservation and model weights.

    Raises
    ------
    ValueError
        Invalid counts, including bools. Do not mutate lengths.
    """
    raise NotImplementedError("TODO: independent request accounting")


In [ ]:
%%lsm
print("Arithmetic fixture checks:", infra.check_account(account_kv))


In [ ]:
%%lsm
ACCOUNT = account_kv([17, 31, 33], 16, BYTES_PER_POSITION)
import pandas as pd
from IPython.display import display

show_table(
    [{"quantity": key, "value": value} for key, value in ACCOUNT.items()],
    ["quantity", "value"],
    "Independent request blocks:17/31/33, block16; byte fields are bytes",
)


### A3. One change: block size

Keep the same lengths and payload dtype. Compare 8, 16, 32. Predict the trade-off: tail slots versus number of table entries. Byte totals below describe ideal block capacity, not measured GPU memory. There is no “best block size” until kernel constraints and real overheads are known.


In [ ]:
%%lsm
BLOCK_SIZES = [8, 16, 32]
BLOCK_RECORDS = [
    dict(block_size=b, **account_kv([17, 31, 33], b, BYTES_PER_POSITION))
    for b in BLOCK_SIZES
]
show_table(
    BLOCK_RECORDS,
    ["block_size", "blocks", "unused_positions", "capacity_bytes"],
    "Same lengths17/31/33; block size in positions, capacity in bytes",
)


In [ ]:
%%lsm
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(
    [str(r["block_size"]) for r in BLOCK_RECORDS],
    [r["unused_positions"] for r in BLOCK_RECORDS],
)
ax.set(
    xlabel="Positions per block",
    ylabel="Unused tail positions",
    title="Same three invented lengths:17,31,33",
)
fig.tight_layout()
plt.show()


## B. Attention over pages is still one attention

Toy D=1, query 1, keys [1,2,3,4], values [10,20,30,40]. All keys are legal for this final query. Predict whether normalizing each two-key page then averaging the two outputs equals the global result. The helper uses stable softmax. A joint permutation of K/V pairs leaves this full-context sum unchanged. Reading another request’s page or breaking K/V pairing is a different failure.

For this scalar toy, scores are $s_j=qk_j$. One attention is $y=\sum_j e^{s_j}v_j/\sum_j e^{s_j}$, over ALL legal keys. Subtracting the same maximum from every score prevents large exponentials without changing the normalized weights.

For page $p$, define $Z_p=\sum_{j\in p}e^{s_j}$ and $y_p=\sum_{j\in p}e^{s_j}v_j/Z_p$. Then $y=(Z_1y_1+Z_2y_2)/(Z_1+Z_2)$, not $(y_1+y_2)/2$. Here the normalized page masses are 0.119203 and 0.880797: the second page must receive more weight. [Read the worked mechanism in Book](https://levshaazz.github.io/llm-serving-mastery/en/book/05/#attention).


In [ ]:
%%lsm
ATTENTION_WEIGHTS, ATTENTION_OUTPUT = infra.scalar_attention(
    [1, 2, 3, 4], [10, 20, 30, 40]
)
show_table(
    [
        dict(position=i, key=i + 1, value=10 * (i + 1), weight=w)
        for i, w in enumerate(ATTENTION_WEIGHTS)
    ],
    ["position", "key", "value", "weight"],
    "Invented D=1 final query; one global softmax",
)


In [ ]:
%%lsm
LOCAL_AVERAGE = (
    infra.scalar_attention([1, 2], [10, 20])[1]
    + infra.scalar_attention([3, 4], [30, 40])[1]
) / 2
show_table(
    [
        {"operation": "global attention", "output": ATTENTION_OUTPUT},
        {"operation": "wrong page-average", "output": LOCAL_AVERAGE},
    ],
    ["operation", "output"],
    "Different normalization gives different outputs",
)


## C1. One short request finishes before its cohort

Two slots. A/B/C arrive at tick 0 with 5/1/1 decode steps; D arrives tick 2 with 2 steps. Each active request executes one decode step per tick. Prefill costs zero ONLY in this toy. A completion at the end of [t,t+1) releases a slot at t+1. Static mode waits for the entire current cohort to empty before refill; continuous mode can refill each tick. Predict C’s completion, then implement arrival-aware FIFO selection. The given lengths create stopping events, not a production oracle for future generated length.


In [ ]:
%%lsm
REQUESTS = infra.REQUESTS
show_table(
    REQUESTS,
    ["id", "arrival", "decode_steps"],
    "Invented requests; time in equal-cost ticks, prefill0",
)


### Task 2 — decide who can start at this tick

Replace only the TODO in `admit_fifo`. `pending` is already FIFO-ordered; do not sort it or mutate it. Return IDs that have arrived, bounded by free slots. An empty slot is not permission to admit a future arrival.

Predict when C can start in each toy mode. Run the following fixture cell, compare both complete traces and read residence as finish minus arrival. Explain why these ticks cannot be interpreted as GPU seconds.


In [ ]:
%%lsm
def admit_fifo(pending: Sequence[Request], tick: int, free_slots: int) -> list[str]:
    """Select arrived requests without modifying the FIFO pending queue.

    Parameters
    ----------
    pending : Sequence[Request]
        Valid requests already in FIFO order; arrival is an integer toy tick.
    tick : int
        Nonnegative boundary at which admission is evaluated.
    free_slots : int
        Nonnegative available decode slots, not memory blocks or token budget.

    Returns
    -------
    list[str]
        Arrived request IDs in queue order, bounded by free_slots.
        Empty/no-arrived/zero-slot inputs return an empty list.

    Raises
    ------
    ValueError
        Invalid tick or slot count, including bools.
    """
    raise NotImplementedError("TODO: arrival-aware FIFO admission")


In [ ]:
%%lsm
print("Admission fixture checks:", infra.check_admission(admit_fifo))


In [ ]:
%%lsm
STATIC = infra.simulate(admit_fifo, REQUESTS, mode="static")
CONTINUOUS = infra.simulate(admit_fifo, REQUESTS, mode="continuous")


In [ ]:
%%lsm
TRACE_MODE = "continuous"
assert TRACE_MODE in ("static", "continuous")
TRACE = STATIC if TRACE_MODE == "static" else CONTINUOUS
show_table(
    [
        {
            "tick": r["tick"],
            "active": ",".join(r["active"]),
            "waiting": ",".join(r["waiting"]),
            "finished_at_next_tick": ",".join(r["finished"]),
        }
        for r in TRACE["ticks"]
    ],
    ["tick", "active", "waiting", "finished_at_next_tick"],
    "Mode " + TRACE_MODE + "; work during[t,t+1)",
)


### Read the trace

One row is work during [tick,tick+1). “Finished” refers to the next boundary, not instant completion at row start. Rerun this viewer with static: it must not recompute or time GPU work. Why is B’s early completion compatible with C still waiting in the static cohort? Which assumption makes the plotted tick widths equal?


In [ ]:
%%lsm
show_table(
    [
        {
            "request": r["id"],
            "arrival_tick": r["arrival"],
            "static_finish": STATIC["completion"][r["id"]],
            "continuous_finish": CONTINUOUS["completion"][r["id"]],
            "static_residence": STATIC["residence_ticks"][r["id"]],
            "continuous_residence": CONTINUOUS["residence_ticks"][r["id"]],
        }
        for r in REQUESTS
    ],
    ["request", "arrival_tick", "static_residence", "continuous_residence"],
    "All four request residences; subtract each arrival",
)


In [ ]:
%%lsm
fig, ax = plt.subplots(figsize=(7, 3))
for y, (name, result) in enumerate([("static", STATIC), ("continuous", CONTINUOUS)]):
    for row in result["ticks"]:
        for lane, request in enumerate(row["active"]):
            ax.broken_barh(
                [(row["tick"], 1)], (y * 3 + lane, 0.8), facecolors="#2A6FDB"
            )
            ax.text(
                row["tick"] + 0.5,
                y * 3 + lane + 0.4,
                request,
                ha="center",
                va="center",
                color="white",
            )
ax.set(
    yticks=[0.9, 3.9],
    yticklabels=["static", "continuous"],
    xlabel="Invented tick; compact active list, NOT persistent physical lanes",
    title="Active work, not GPU utilization; prefill omitted",
)
ax.set_xticks(range(8))
fig.tight_layout()
plt.show()


## D. Explain before measuring

The real V1 scheduler has a scheduled-token budget, prefill work, KV capacity and possible preemption. “Four clients”, “four active sequences”, “four new tokens” and “four KV blocks” are not interchangeable. A 256-token budget can include several decode positions and a prefill chunk. This toy demonstrates refill logic, not a quantitative performance model of vLLM.

For the real comparison, keep Qwen revision, dtype, exact rendered token IDs, request population, output cap/EOS rule and warmup fixed. Disable prefix caching in the control. Submitted offline burst size 1 versus 4 is the student change. Pipeline versus vLLM is a whole-stack comparison, not a PagedAttention ablation. Non-streaming cohort wall time cannot establish TTFT or a p95 from three repeats. All failed and semantically wrong requests stay in the denominator.


In [ ]:
%%lsm
EXPLANATION = {
    "why_independent_rounding": "TODO: identify the request boundary",
    "why_global_softmax": "TODO: use the two computed outputs",
    "why_ticks_not_seconds": "TODO: name two omitted real costs",
    "what_pipeline_vs_vllm_proves": "TODO: distinguish stack comparison from mechanism causality",
}


## Preserve your work

Keep your modified notebook, block table, both complete traces and explanations. CPU work is not GPU completion. Do not invent measurements or reuse Topic00’s 64-client figures as a Topic05 result. Instructor solutions and actual execution captures remain private.


## E. Real model: offline bursts, not HTTP clients

Compare pipeline with vLLM on the same twelve literal tickets, frozen Qwen revision, FP16, natural EOS and output cap 16. Change the submitted burst cohort from 1 to 4. This library comparison does not measure client TTFT or individual E2E latency. It changes more than PagedAttention. Prefix caching is disabled. Three repeats per size/engine retain 144 attempted predictions and 90 cohort intervals. Warmup is outside the measured population. Cohort-size order alternates by repeat.

This experiment does NOT observe a fifth waiting request entering a freed slot: each submitted burst has at most four requests and finishes before the next call. Continuous refill was demonstrated by the toy, not empirically isolated here. Both engines explicitly use repetition penalty 1.0; greedy alone would not override Qwen’s inherited penalty 1.1.

This workload targets standard-RAM T4; see the draft acceptance notice above. Workload-specific bounds: 6 GiB free host and device before load; 2 GiB free host / 512 MiB free device while working. These do not certify free-account availability or change judge/operator gates. Do not install provider-kernel pip packages. Keep the exact managed runtime.


In [ ]:
HELPER_URL = "https://raw.githubusercontent.com/levshaazz/llm-serving-mastery-submission-example/d81c299ac1c8fe6249f8b33fa49cd7d42add42ea/teaching/topic05_gpu.py"
HELPER_SHA = "eaabc08918229d95606a8429ad4a2757706e6ea67e09620a271f438210d42381"
helper_bytes = download_verified(HELPER_URL, HELPER_SHA, 128 * 1024)
helper_path = RUNTIME_ROOT / "workspace" / "topic05_gpu.py"
if helper_path.exists():
    assert hashlib.sha256(helper_path.read_bytes()).hexdigest() == HELPER_SHA
else:
    with helper_path.open("xb") as destination:
        destination.write(helper_bytes)
print("Verified public helper:", helper_path.name, "; no inference")


In [ ]:
%%lsm
GPU_HELPER = Path("topic05_gpu.py")
gpu_spec = importlib.util.spec_from_file_location("topic05_gpu", GPU_HELPER)
gpu = importlib.util.module_from_spec(gpu_spec)
gpu_spec.loader.exec_module(gpu)


In [ ]:
%%lsm
show_table(
    [{"case": case, "ticket": text, "gold": gold} for case, text, gold in gpu.CASES],
    ["case", "ticket", "gold"],
    "Twelve literal ML inputs; balanced gold labels, not promised predictions",
)


### Predict before installing GPU packages

Write a falsifiable expectation: which engine/size might process the same twelve tickets faster, and why? Predict whether literal labels must be identical. State one observation that would contradict your prediction. These are hypotheses, not measured answers. In quality, also state a concrete pre-run classroom acceptance/refusal rule using correct/12, invalid/12 and the pipeline control. Do not change that rule after seeing results. Passing this small screen never authorizes a production recommendation. Save the notebook before installing or running inference. The prediction variable is not automatically locked; do not rewrite it after observing results.


In [ ]:
%%lsm
PREDICTION = {
    "rate": "TODO: predict the direction and mechanism, not a fabricated number",
    "quality": "TODO: can equal greedy settings still disagree?",
    "falsifier": "TODO: name an observation that would contradict your expectation",
}


### Task 3 — one controlled real-model change

Finish your CPU code and `EXPLANATION`, write `PREDICTION`, and save the notebook BEFORE inference. Add 4 to `COHORT_SIZES`, retaining 1; leave `REPEATS=3`. Set `RUN_GPU=True` when you are ready for the full mandatory experiment. Leaving it False stops at the GPU gate, not at a successful completion.

Keep model, revision, FP16, all twelve tickets, chat template, greedy settings, output cap, EOS and disabled prefix caching unchanged. A burst of four submitted library inputs is not four HTTP clients. Expected population: 2 engines × 2 sizes × 3 repeats × 12 tickets = 144 predictions, but only 12 distinct public tickets.


In [ ]:
%%lsm
COHORT_SIZES = [1]  # TODO: retain1 and add4; no other workload factor changes.
REPEATS = 3
RUN_GPU = False  # Set True only after explaining the CPU results and reading stops.


In [ ]:
%%lsm
def verify_ready() -> gpu.Headroom:
    """Recheck current student work immediately before installation or inference.

    Returns
    -------
    Headroom
        Fresh provisional resource sample after all current prerequisites pass.

    Raises
    ------
    AssertionError
        Missing, invalid or still-TODO explanations, prediction or workload.
    RuntimeError
        Insufficient resources. No installation or model inference starts.
    """
    infra.check_account(account_kv)
    infra.check_admission(admit_fifo)
    assert COHORT_SIZES == [1, 4] and all(type(n) is int for n in COHORT_SIZES)
    assert set(EXPLANATION) == {
        "why_independent_rounding",
        "why_global_softmax",
        "why_ticks_not_seconds",
        "what_pipeline_vs_vllm_proves",
    }
    assert set(PREDICTION) == {"rate", "quality", "falsifier"}
    assert all(
        isinstance(v, str) and v.strip() and "TODO" not in v.upper()
        for v in [*EXPLANATION.values(), *PREDICTION.values()]
    )
    return gpu.headroom(True)


In [ ]:
%%lsm
if RUN_GPU:
    print("Observed pre-install headroom:", verify_ready())
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


In [ ]:
opt_in = bridge.execute("print(RUN_GPU)")
assert opt_in["ok"] and opt_in["stdout"].strip() in ("True", "False")
if opt_in["stdout"].strip() == "True":
    prereq = bridge.execute("verify_ready()")
    assert prereq["ok"], (
        "Stop: current CPU/prediction/resource gate failed; "
        + prereq.get("error", "")
        + " "
        + prereq.get("stderr", "")
    )
    RUNTIME_PYTHON = runtime.install_runtime(RUNTIME_ROOT, "serving")
    print("Frozen GPU serving extras installed in managed worker environment")
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


In [ ]:
%%lsm
from datetime import datetime, timezone

GPU_ATTEMPT = None
if RUN_GPU:
    verify_ready()
    GPU_ATTEMPT = Path("evidence") / datetime.now(timezone.utc).strftime(
        "topic05-%Y%m%dT%H%M%S%fZ"
    )
    GPU_RECEIPT = gpu.run_comparison(GPU_ATTEMPT, COHORT_SIZES, REPEATS)
    print("Two actual owned children completed; preserve", str(GPU_ATTEMPT))
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


### Did the model actually predict?

The next check reads THIS run's files and requires 144 predictions and 90 cohort intervals across both engines. A green execution counter does not establish this. Zero/partial predictions must fail, never be replaced by historical outputs. All wrong or invalid answers remain in the attempted population. No inference happens in any viewer.


In [ ]:
%%lsm
def require_gpu_result() -> Path:
    """Require this notebook's actual GPU attempt, not a successful no-op.

    Returns
    -------
    Path
        Current attempt directory. Completeness is checked by gpu.load_pair.

    Raises
    ------
    RuntimeError
        GPU was disabled, inference never ran, or the current directory is absent.
        Preserve a failed attempt; never fabricate outputs or load an old run.
    """
    attempt = globals().get("GPU_ATTEMPT")
    if globals().get("RUN_GPU") is not True or not isinstance(attempt, Path):
        raise RuntimeError(
            "No predictions. Enable RUN_GPU and execute the GPU installation/run cells first."
        )
    if not attempt.is_dir():
        raise RuntimeError(
            "Current GPU attempt directory is absent; preserve the failure and use a fresh full run."
        )
    return attempt


In [ ]:
%%lsm
GPU_PAIR = gpu.load_pair(require_gpu_result())
COUNT_ROWS = [
    {
        "engine": engine,
        "predictions": len(data["requests"]),
        "cohort_intervals": len(data["cohorts"]),
    }
    for engine, data in GPU_PAIR.items()
]
assert sum(row["predictions"] for row in COUNT_ROWS) == 144
assert sum(row["cohort_intervals"] for row in COUNT_ROWS) == 90
show_table(
    COUNT_ROWS,
    ["engine", "predictions", "cohort_intervals"],
    "Actual full workload: 72 predictions and 45 cohort intervals per engine; no skipped run is complete",
)


### What to inspect in the preserved files

Each engine keeps its rendered token ledger, 45 cohort intervals and 72 individual prediction rows. Across both engines this is 90 intervals and 144 predictions. All failures remain partial, never reconstructed as complete. Pipeline native tensors can include prompt/padding; the helper preserves them separately from trimmed selected output IDs. Rate accounting excludes terminal configured EOS, with actual API differences retained. Empty completion is a wrong/invalid task response, not an excuse to drop an attempted request.

Compare each case_id, rendered text and input_ids in the two prompt ledgers, not just the visible ticket. Inspect model revision, dtype and generation_profile (greedy, repetition penalty, output cap and EOS). Pipeline native full tensors are not completion IDs; check the declared native-output scope before counting work.


In [ ]:
%%lsm
import json

GPU_VIEW_ENGINE = "vllm"
GPU_VIEW_SIZE = 4
GPU_VIEW_REPEAT = 0
assert GPU_VIEW_ENGINE in ("pipeline", "vllm")
if GPU_ATTEMPT is not None:
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    GPU_DATA = json.loads((GPU_ATTEMPT / (GPU_VIEW_ENGINE + ".json")).read_text())
    gpu.validate_backend(GPU_DATA, GPU_VIEW_ENGINE, COHORT_SIZES, REPEATS)
    show_table(
        [
            {"case_ids": ",".join(r["case_ids"]), "cohort_wall_ms": r["wall_s"] * 1000}
            for r in GPU_DATA["cohorts"]
            if r["cohort_size"] == GPU_VIEW_SIZE and r["repeat"] == GPU_VIEW_REPEAT
        ],
        ["case_ids", "cohort_wall_ms"],
        "Actual "
        + GPU_VIEW_ENGINE
        + " burst"
        + str(GPU_VIEW_SIZE)
        + ", repeat"
        + str(GPU_VIEW_REPEAT)
        + "; all90intervals retained in both files",
    )
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


In [ ]:
%%lsm
import json

if GPU_ATTEMPT is not None:
    assert GPU_VIEW_ENGINE in ("pipeline", "vllm")
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    OUTPUT_DATA = json.loads((GPU_ATTEMPT / (GPU_VIEW_ENGINE + ".json")).read_text())
    gpu.validate_backend(OUTPUT_DATA, GPU_VIEW_ENGINE, COHORT_SIZES, REPEATS)
    show_table(
        [
            {
                "case": r["case_id"],
                "gold": r["gold"],
                "literal_output": r["output"],
                "correct": r["correct"],
            }
            for r in OUTPUT_DATA["requests"]
            if r["cohort_size"] == GPU_VIEW_SIZE and r["repeat"] == GPU_VIEW_REPEAT
        ],
        ["case", "gold", "literal_output", "correct"],
        "Actual "
        + GPU_VIEW_ENGINE
        + "; twelve literal outputs for selected size/repeat; all attempts retained",
    )
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


### Compare a fixed ML population, not convenient clocks

For each engine/size/repeat compute 12 divided by the sum of its cohort wall seconds. The provided reducer does this; do not write timing statistics. Read all three repetitions separately. Compare correct/12 and invalid/12 before recommending a faster route. This tiny repeated population cannot certify generalization or p95. A failed/partial engine has no complete rate.


In [ ]:
%%lsm
if GPU_ATTEMPT is not None:
    show_table(
        gpu.summarize(gpu.load_pair(GPU_ATTEMPT)),
        [
            "engine",
            "cohort_size",
            "repeat",
            "attempted",
            "wall_s",
            "requests_per_s",
            "correct",
            "invalid",
        ],
        "All twelve attempts per row; rate12/sum(cohort clocks); three repeated populations, not independent datasets",
    )
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


In [ ]:
%%lsm
if GPU_ATTEMPT is not None:
    RATE_ROWS = gpu.summarize(gpu.load_pair(GPU_ATTEMPT))
    fig, ax = plt.subplots(figsize=(7, 3))
    for i, (engine, size) in enumerate(
        [("pipeline", 1), ("pipeline", 4), ("vllm", 1), ("vllm", 4)]
    ):
        rates = [
            r["requests_per_s"]
            for r in RATE_ROWS
            if r["engine"] == engine and r["cohort_size"] == size
        ]
        ax.scatter([i] * 3, rates, color="#2A6FDB", s=40)
    ax.set(
        xticks=range(4),
        xticklabels=["pipeline1", "pipeline4", "vLLM1", "vLLM4"],
        ylabel="Attempts / synchronized library second",
        title="Actual three repeat rates; output correctness checked separately",
    )
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    plt.show()
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


### Read the three measured repetitions

Very close dots can overlap; confirm all three values in the preceding 12-row table. Rate means 12 attempts divided by the sum of synchronized cohort seconds, not a mean of individual reciprocals. Compare each engine's 1→4 change, then compare engines at the SAME size. Is the winner the same in every repeat? Describe a result that would falsify your original prediction. Three repetitions cannot establish p95, significance or production capacity.


### Count first, then draw

The next cell builds both 3×4 matrices from the actual selected size/repeat, retaining all 12 attempts including invalid outputs. It is silent: no GPU inference or figure. The following cell draws one paired figure with gold rows and predicted columns. If you change the selected attempt/size/repeat, rebuild the counts before drawing; the plot rejects a stale selection.


In [ ]:
%%lsm
require_gpu_result()
import numpy as np

QUALITY_SELECTION = None
QUALITY_MATRICES = {}
labels = ["billing", "access", "bug"]
predicted = labels + ["invalid"]
if GPU_ATTEMPT is not None:
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    QUALITY_PAIR = gpu.load_pair(GPU_ATTEMPT)
    for engine, data in QUALITY_PAIR.items():
        matrix = np.zeros((3, 4), dtype=int)
        for row in data["requests"]:
            if row["cohort_size"] == GPU_VIEW_SIZE and row["repeat"] == GPU_VIEW_REPEAT:
                guess = row["output"].strip().lower()
                column = predicted.index(guess) if guess in labels else 3
                matrix[labels.index(row["gold"]), column] += 1
        assert int(matrix.sum()) == 12
        QUALITY_MATRICES[engine] = matrix
    QUALITY_SELECTION = (GPU_ATTEMPT, GPU_VIEW_SIZE, GPU_VIEW_REPEAT)


In [ ]:
%%lsm
require_gpu_result()
if GPU_ATTEMPT is not None:
    assert QUALITY_SELECTION == (
        GPU_ATTEMPT,
        GPU_VIEW_SIZE,
        GPU_VIEW_REPEAT,
    ), "Recompute quality counts after changing the selected attempt/size/repeat"
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    for ax, (engine, matrix) in zip(axes, QUALITY_MATRICES.items()):
        ax.imshow(matrix, cmap="Blues", vmin=0, vmax=4)
        for y in range(3):
            for x in range(4):
                rgb = plt.get_cmap("Blues")(matrix[y, x] / 4)[:3]
                linear = [
                    v / 12.92 if v <= 0.04045 else ((v + 0.055) / 1.055) ** 2.4
                    for v in rgb
                ]
                luminance = sum(v * w for v, w in zip(linear, (0.2126, 0.7152, 0.0722)))
                ink = (
                    "black"
                    if (luminance + 0.05) / 0.05 >= 1.05 / (luminance + 0.05)
                    else "white"
                )
                ax.text(x, y, str(matrix[y, x]), ha="center", va="center", color=ink)
        ax.set(
            xticks=range(4),
            xticklabels=predicted,
            yticks=range(3),
            yticklabels=labels,
            xlabel="Predicted; invalid remains an attempt",
            ylabel="Gold",
            title=engine + "; 12 actual attempts",
        )
    fig.suptitle(
        "Selected burst " + str(GPU_VIEW_SIZE) + ", repeat " + str(GPU_VIEW_REPEAT)
    )
    fig.tight_layout()
    plt.show()
else:
    raise RuntimeError("No predictions: run the full GPU experiment before plotting")


### Is the model useful for routing tickets?

Read gold ROWS and predicted COLUMNS. There are four tickets per gold class in the selected run. Always predicting billing gets 4/12 overall, even though billing recall is 1. Read the next six-row table: which class suffers, and does an overall score conceal it? This is a tiny public smoke population, not held-out generalization.


In [ ]:
%%lsm
CLASS_PAIR = gpu.load_pair(require_gpu_result())
assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
CLASS_ROWS = []
for engine, data in CLASS_PAIR.items():
    selected = [
        r
        for r in data["requests"]
        if r["cohort_size"] == GPU_VIEW_SIZE and r["repeat"] == GPU_VIEW_REPEAT
    ]
    assert len(selected) == 12
    for label in ("billing", "access", "bug"):
        rows = [r for r in selected if r["gold"] == label]
        assert len(rows) == 4
        correct = sum(r["correct"] for r in rows)
        CLASS_ROWS.append(
            {
                "engine": engine,
                "gold_class": label,
                "correct": correct,
                "attempted": len(rows),
                "recall": correct / len(rows),
            }
        )
show_table(
    CLASS_ROWS,
    ["engine", "gold_class", "correct", "attempted", "recall"],
    "Selected size/repeat: recall keeps invalid outputs in the denominator; always-billing baseline is 4/12 overall",
)


In [ ]:
%%lsm
if GPU_ATTEMPT is not None:
    assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
    assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
    DISAGREEMENT_PAIR = gpu.load_pair(GPU_ATTEMPT)
    matching = {
        engine: {
            r["case_id"]: r
            for r in data["requests"]
            if r["cohort_size"] == GPU_VIEW_SIZE and r["repeat"] == GPU_VIEW_REPEAT
        }
        for engine, data in DISAGREEMENT_PAIR.items()
    }
    show_table(
        [
            {
                "case": case,
                "gold": gold,
                "pipeline": matching["pipeline"][case]["output"],
                "vllm": matching["vllm"][case]["output"],
                "literal_equal": matching["pipeline"][case]["output"]
                == matching["vllm"][case]["output"],
            }
            for case, text, gold in gpu.CASES
        ],
        ["case", "gold", "pipeline", "vllm", "literal_equal"],
        "All twelve matched literal outputs; equality is not correctness",
    )
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


### Read a mistake, then propose a test

The next table joins each actual wrong answer to its literal ticket and gold label. Check whether login words dominate an actual bug description; that is a hypothesis, not an established model mechanism. Compare the two engines on the same case before blaming batching. Propose one future matched-prompt calibration test, fix the candidate before held-out evaluation, and keep this run unchanged. No prompt tuning or extra inference happens in this notebook's error viewer. If the table has zero rows, report no observed errors rather than inventing an example; the preceding full-run check still requires all 144 predictions.

Validity/correctness use the declared `strip().lower()` normalization; the table preserves the original output string. A valid class label can still be semantically wrong. Literal engine equality is neither correctness nor evidence of generalization.


In [ ]:
%%lsm
ERROR_PAIR = gpu.load_pair(require_gpu_result())
assert type(GPU_VIEW_SIZE) is int and GPU_VIEW_SIZE in COHORT_SIZES
assert type(GPU_VIEW_REPEAT) is int and GPU_VIEW_REPEAT in range(REPEATS)
ticket_by_id = {case: text for case, text, gold in gpu.CASES}
ERROR_ROWS = [
    {
        "engine": engine,
        "case": row["case_id"],
        "ticket": ticket_by_id[row["case_id"]],
        "gold": row["gold"],
        "literal_output": row["output"],
    }
    for engine, data in ERROR_PAIR.items()
    for row in data["requests"]
    if row["cohort_size"] == GPU_VIEW_SIZE
    and row["repeat"] == GPU_VIEW_REPEAT
    and not row["correct"]
]
show_table(
    ERROR_ROWS,
    ["engine", "case", "ticket", "gold", "literal_output"],
    str(len(ERROR_ROWS))
    + " actual incorrect answers in the selected paired 24 attempts; zero errors is not zero predictions",
)


### Your engineering decision

Use your previously written prediction, all three rates, all-attempt quality and one literal error/disagreement if any. If none exist, inspect a correct matching answer and explicitly report zero errors/disagreements; never invent one. Save the notebook before inference; this notebook does not automatically lock the prediction. Do not rewrite it after observing results. Choose a route for this measured workload or explicitly refuse to recommend one. Do not recommend deployment if quality fails your task requirement. The small balanced corpus is a smoke test, not a held-out production evaluation. A throughput change here cannot be attributed specifically to PagedAttention or continuous refill.


### Task 4 — write your bounded engineering decision

Fill the existing four `DECISION` fields; no extra model run is needed.

- `choice`: choose a route for THIS workload, or refuse a recommendation. Compare its quality to your pre-run rule, not only its speed.
- `evidence`: quote all three relevant rates with units, correct/12 and invalid/12; identify one case, its full ticket, gold and literal output. Name the weakest class and its recall numerator/denominator.
- `alternative_explanation`: distinguish library/kernel differences from an isolated PagedAttention intervention. Compare both engines on your selected case before blaming the runtime.
- `limitation`: state a falsifiable hypothesis about a real error, a future matched calibration test and an untouched held-out check. If no errors occur, use a real correct case and state that limitation.

A hypothesis is not an observed mechanism. Do not rewrite the prediction or tune this already-observed smoke corpus and call it held-out evidence.


In [ ]:
%%lsm
DECISION = {
    "choice": "TODO: route or refuse recommendation for the measured workload",
    "evidence": "TODO: quote actual repeat rates, quality counts and a literal example",
    "alternative_explanation": "TODO: another stack difference, not isolated PagedAttention",
    "limitation": "TODO: state an untested requirement and the next experiment",
}


In [ ]:
%%lsm
import json

if GPU_ATTEMPT is not None:
    assert set(DECISION) == {
        "choice",
        "evidence",
        "alternative_explanation",
        "limitation",
    }, "Complete the four required decision fields"
    assert all(
        (
            isinstance(v, str) and v.strip() and ("TODO" not in v.upper())
            for v in DECISION.values()
        )
    )
    (GPU_ATTEMPT / "worksheet.json").write_text(
        json.dumps(
            {
                "prediction": PREDICTION,
                "decision": DECISION,
                "cpu_explanation": EXPLANATION,
            },
            ensure_ascii=False,
            indent=2,
        )
        + "\n"
    )
    print("Saved your worksheet; raw engine evidence unchanged")
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


In [ ]:
%%lsm
from IPython.display import HTML, display

if GPU_ATTEMPT is not None:
    display(HTML(gpu.archive_attempt(GPU_ATTEMPT)))
else:
    raise RuntimeError(
        "GPU investigation is incomplete: no predictions. Use a T4 runtime, set RUN_GPU=True, and run the GPU stages. CPU preparation is not completion."
    )


Download the actual attempt ZIP and your modified notebook before disconnecting. Preserve any partial attempt separately. Do not export model caches or credentials. Results and instructor solutions stay private. A cohort comparison cannot certify deployment quality, p95, TTFT or learner transfer.


In [ ]:
CLOSE_RUNTIME = False  # Set True only after saving all needed evidence.
if CLOSE_RUNTIME:
    bridge.close()
    assert bridge.process.poll() == 0
    print("Owned worker exited; preserve files before disconnecting the VM")
else:
    print(
        "Worker remains available for viewer cells. Save evidence before final closure."
    )
